In [13]:
import pandas as pd
import sqlite3

# 1. Conexión: La base de datos se creará en el archivo 'SoporteTI_DB.db'
#    en la misma carpeta donde ejecutas el Notebook.
conn = sqlite3.connect('SoporteTI_DB.db')
cursor = conn.cursor()

In [14]:
# Consulta 1: Creación de la Tabla de Usuarios
sql_create_users_table = """
CREATE TABLE IF NOT EXISTS Usuarios (
    UsuarioID TEXT PRIMARY KEY,
    NombreCompleto TEXT NOT NULL,
    Rol TEXT,
    Departamento TEXT,
    Activo BOOLEAN,
    UltimoLogin DATE
);
"""

# Consulta 2: Creación de la Tabla de Tickets
sql_create_tickets_table = """
CREATE TABLE IF NOT EXISTS Tickets (
    TicketID TEXT PRIMARY KEY,
    UsuarioID TEXT,
    FechaCreacion DATE,
    FechaCierre DATE,
    Categoria TEXT,
    Prioridad TEXT,
    TecnicoID TEXT,
    SLA_Horas INTEGER,
    TiempoResolucionHoras REAL,
    Estado TEXT,
    FOREIGN KEY (UsuarioID) REFERENCES Usuarios(UsuarioID)
);
"""

In [15]:
# Ejecutar las consultas de creación de tablas
cursor.execute(sql_create_users_table)
cursor.execute(sql_create_tickets_table)

# Guardar los cambios
conn.commit()

print("Tablas Usuarios y Tickets creadas en 'SoporteTI_DB.db' exitosamente.")

Tablas Usuarios y Tickets creadas en 'SoporteTI_DB.db' exitosamente.


In [16]:
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS usuarios;")
cursor.execute("DROP TABLE IF EXISTS Tickets;")
conn.commit()

In [17]:
# --- Carga y Limpieza con Pandas ---

# Asegúrate de que los archivos CSV estén en la misma carpeta.
df_users = pd.read_csv('../data/raw/usuarios.csv', encoding='latin1')
df_tickets = pd.read_csv('../data/raw/tickets.csv', encoding='latin1')

# 1. Conversión de tipos de datos en Pandas antes de la carga
#    En tickets, aseguramos que las fechas se carguen correctamente.
df_tickets['FechaCreacion'] = pd.to_datetime(df_tickets['FechaCreacion']).dt.strftime('%Y-%m-%d')
df_tickets['FechaCierre'] = pd.to_datetime(df_tickets['FechaCierre']).dt.strftime('%Y-%m-%d')

#    En usuarios, limpiamos la fecha de login.
df_users['UltimoLogin'] = pd.to_datetime(df_users['UltimoLogin']).dt.strftime('%Y-%m-%d')

# 2. Carga a SQLite
#    if_exists='replace' borra la tabla y la recrea si ya existe (útil para desarrollo).
df_users.to_sql('usuarios', conn, if_exists='replace', index=False)
df_tickets.to_sql('tickets', conn, if_exists='replace', index=False)

print("Datos cargados de CSV a las tablas de SQLite exitosamente.")

# ¡La base de datos ya está lista para el análisis SQL!

Datos cargados de CSV a las tablas de SQLite exitosamente.


In [18]:
import sqlite3
import pandas as pd

# Volver a abrir la conexión
conn = sqlite3.connect("SoporteTI_DB.db")

sql_query_ranking = """
SELECT * FROM Tickets LIMIT 5
"""

df_resultado_sql = pd.read_sql(sql_query_ranking, conn)
print(df_resultado_sql)

conn.close()

  TicketID UsuarioID FechaCreacion FechaCierre Categoria Prioridad TecnicoID  \
0     T001      U003    2025-10-01  2025-10-01    Acceso      Alta     Tec01   
1     T002      U002    2025-10-01  2025-10-02  Software     Media     Tec02   
2     T003      U003    2025-10-02  2025-10-04  Hardware      Alta     Tec01   
3     T004      U004    2025-10-03  2025-10-03  Software      Baja     Tec03   
4     T005      U001    2025-10-04  2025-10-04       Red      Alta     Tec02   

   SLA_Horas  TiempoResolucionHoras   Estado  
0          4                    1.5  Cerrado  
1         24                   18.0  Cerrado  
2          4                   48.0  Cerrado  
3         48                    2.0  Cerrado  
4          4                    3.0  Cerrado  


In [19]:
import pandas as pd
import sqlite3
import os

# --- Configuración de Rutas ---
# Definición de las rutas de archivo, asumiendo que el notebook está en 'proyecto_soporte_ti/'
BASE_DIR = 'data'
RAW_PATH = os.path.join('..', 'data', 'raw')
PROCESSED_PATH = os.path.join('..', 'data', 'processed')
DB_FILE = os.path.join(PROCESSED_PATH, 'SoporteTI_DB.db')
OUTPUT_FILE = os.path.join(PROCESSED_PATH, 'analisis_ti_processed.csv')

# Asegurarse de que el directorio 'processed' exista
os.makedirs(PROCESSED_PATH, exist_ok=True)

In [20]:
# --- 1. Conexión a la Base de Datos ---
conn = sqlite3.connect("SoporteTI_DB.db")
cursor = conn.cursor()

print("--- Paso 1: Conexión y Creación de Tablas ---")

# DDL: Creación de la Tabla de Usuarios
sql_create_users_table = """
CREATE TABLE IF NOT EXISTS Usuarios (
    UsuarioID TEXT PRIMARY KEY,
    NombreCompleto TEXT NOT NULL,
    Rol TEXT,
    Departamento TEXT,
    Activo BOOLEAN,
    UltimoLogin DATE
);
"""

# DDL: Creación de la Tabla de Tickets
sql_create_tickets_table = """
CREATE TABLE IF NOT EXISTS Tickets (
    TicketID TEXT PRIMARY KEY,
    UsuarioID TEXT,
    FechaCreacion DATE,
    FechaCierre DATE,
    Categoria TEXT,
    Prioridad TEXT,
    TecnicoID TEXT,
    SLA_Horas INTEGER,
    TiempoResolucionHoras REAL,
    Estado TEXT,
    FOREIGN KEY (UsuarioID) REFERENCES Usuarios(UsuarioID)
);
"""

cursor.execute(sql_create_users_table)
cursor.execute(sql_create_tickets_table)
conn.commit()
print("Tablas Usuarios y Tickets creadas en la base de datos.")

--- Paso 1: Conexión y Creación de Tablas ---
Tablas Usuarios y Tickets creadas en la base de datos.


In [11]:
import sqlite3
import pandas as pd
import os

# Volver a abrir la conexión
conn = sqlite3.connect("SoporteTI_DB.db")

# --- 2. Carga y Limpieza (ETL) con Pandas ---
print("\n--- Paso 2: Carga y Limpieza con Pandas ---")

try:
    df_users = pd.read_csv(os.path.join(RAW_PATH, 'usuarios.csv'), encoding='latin-1')
    df_tickets = pd.read_csv(os.path.join(RAW_PATH, 'tickets.csv'), encoding='latin-1')
except FileNotFoundError:
    print("¡ERROR! Asegúrate de que los archivos 'usuarios.csv' y 'tickets.csv' estén en la carpeta:", RAW_PATH)
    conn.close()
    exit()

# Limpieza: Asegurar formato de fecha antes de la carga a SQLite
df_tickets['FechaCreacion'] = pd.to_datetime(df_tickets['FechaCreacion']).dt.strftime('%Y-%m-%d')
df_tickets['FechaCierre'] = pd.to_datetime(df_tickets['FechaCierre']).dt.strftime('%Y-%m-%d')
df_users['UltimoLogin'] = pd.to_datetime(df_users['UltimoLogin']).dt.strftime('%Y-%m-%d')

# Carga a SQLite
df_users.to_sql('usuarios', conn, if_exists='replace', index=False)
df_tickets.to_sql('tickets', conn, if_exists='replace', index=False)
print("Datos cargados a SQLite listos para el análisis.")


--- Paso 2: Carga y Limpieza con Pandas ---


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf3 in position 178: invalid continuation byte

In [ ]:
# --- 3. Consulta SQL Intermedia (Análisis y Extracción de Insight) ---
print("\n--- Paso 3: Ejecución de Consulta SQL Intermedia ---")

# Query Intermedia: Análisis de SLA (Acuerdo de Nivel de Servicio) y Categoría
# Utiliza un CASE STATEMENT para determinar el incumplimiento del SLA y JOINs para unir datos del usuario.
sql_query_sla = """
SELECT
    t.Categoria,
    u.Departamento,
    COUNT(t.TicketID) AS TotalTickets,
    -- 1. Uso de CASE STATEMENT para marcar el incumplimiento del SLA (Habilidad Intermedia)
    SUM(CASE WHEN t.TiempoResolucionHoras > t.SLA_Horas THEN 1 ELSE 0 END) AS TicketsFueraSLA,
    -- 2. Cálculo del Porcentaje de Incumplimiento
    ROUND(CAST(SUM(CASE WHEN t.TiempoResolucionHoras > t.SLA_Horas THEN 1 ELSE 0 END) AS REAL) * 100 / COUNT(t.TicketID), 2) AS PorcentajeFueraSLA
FROM
    Tickets t
JOIN
    Usuarios u ON t.UsuarioID = u.UsuarioID
GROUP BY
    t.Categoria, u.Departamento
ORDER BY
    PorcentajeFueraSLA DESC, TotalTickets DESC;
"""

# Ejecutar la consulta y guardar el resultado en un DataFrame de Pandas
df_analisis_sla = pd.read_sql(sql_query_sla, conn)

print("Consulta ejecutada. Primeros 5 resultados del análisis de SLA por Categoría y Departamento:")
print(df_analisis_sla.head())

In [ ]:
# --- 4. Exportación del Resultado a la Ruta Procesada ---
print("\n--- Paso 4: Exportación para Power BI ---")

# Exportar el DataFrame resultante a la ruta '../data/processed/analisis_ti_processed.csv'
df_analisis_sla.to_csv(OUTPUT_FILE, index=False)
print(f"Resultado exportado a: {OUTPUT_FILE}")

# --- Cierre de la Conexión ---
conn.close()
print("Conexión a la base de datos cerrada.")

# --- Indicación para Power BI ---
print("\n--- ¡LISTO PARA POWER BI! ---")
print("Puedes importar el archivo CSV exportado desde Power BI usando 'Obtener datos' -> 'Texto/CSV'.")